In [1]:
# router_prompt_learning.py
"""
使用 TextGrad + TPD-AHD 思想，为 AgentBasedRouter 学习一段更好的
“模型能力描述”（只改写 prompt 中 [DEEPSEEK]/[QWEN] 那部分）。

整体流程（对应 TPD-AHD 的 forward / backward）：

1. 初始化一组 router-prompt 候选（population），变量是
   - deepseek_desc
   - qwen_desc

2. 对每个候选：
   - 用对应的描述实例化 AgentBasedRouter
   - 在 OBP + EoH 上跑一小轮，拿到 best_score 作为 fitness
   （EoH 的 fitness 定义与原论文一致：越大越好）

3. 排序得到当前最优候选 h1，构造 best-anchored preference pairs：
   (h1, h2), (h1, h3), ... (h1, hN)，对应 TPD-AHD 中的偏好配对机制。

4. 对每一对 (h1, hk)，调用一个“teacher LLM”：
   - 比较两个 router prompt 的描述 + 分数，生成 textual loss（解释为何 h1 更好）。
   - 再生成 textual gradient：给出“如何修改 hk 的 deepseek_desc / qwen_desc 才能更接近 h1，
     同时保留/探索一些新的假设维度”。这一点对应 TextGrad 中通过自然语言反馈
     回传梯度信号的思想。

5. 用 textual gradient 直接让 LLM 产出新的 (deepseek_desc', qwen_desc')，形成下一代 population。

6. 重复若干代，最后选出平均性能最好的 router prompt 作为论文中的“学习型 router”。
"""

import json
import os
import statistics
import time
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional

from llm4ad.task.optimization.online_bin_packing import OBPEvaluation
from llm4ad.tools.llm.llm_api_https import HttpsApi
from llm4ad.method.eoh import EoH, EoHProfiler

from lagent_router import AgentBasedRouter, DEFAULT_DEEPSEEK_DESC, DEFAULT_QWEN_DESC



In [4]:
os.environ["DEEPSEEK_HOST"] = "api.deepseek.com"
os.environ["DEEPSEEK_KEY"] = "sk-457d831f3fd24603ad514f9f26dbe132"
os.environ["DEEPSEEK_MODEL"] = "deepseek-chat"

os.environ["QWEN_HOST"] = "dashscope.aliyuncs.com"
os.environ["QWEN_KEY"] = "sk-f622faad8dee4b179d1c8593b1dab866"
os.environ["QWEN_MODEL"] = "qwen-flash"

In [6]:

# ===================== 数据结构 =====================

@dataclass
class RouterPromptCandidate:
    cid: int
    deepseek_desc: str
    qwen_desc: str
    # 评估得到的 EoH best_score（越大越好）
    score: Optional[float] = None
    # 额外统计信息（调用次数等）
    meta: Optional[Dict[str, Any]] = None


# ===================== 一些小工具 =====================

def build_https_llm_from_env(prefix: str) -> HttpsApi:
    """
    根据环境变量构造一个 HttpsApi：
    - {PREFIX}_HOST
    - {PREFIX}_KEY
    - {PREFIX}_MODEL

    例如：
    - DEEPSEEK_HOST, DEEPSEEK_KEY, DEEPSEEK_MODEL
    - QWEN_HOST, QWEN_KEY, QWEN_MODEL
    - ROUTER_HOST, ROUTER_KEY, ROUTER_MODEL
    - TEACHER_HOST, TEACHER_KEY, TEACHER_MODEL
    """
    host = os.environ.get(f"{prefix}_HOST")
    key = os.environ.get(f"{prefix}_KEY")
    model = os.environ.get(f"{prefix}_MODEL")

    if host is None or key is None or model is None:
        raise ValueError(
            f"Missing env vars for {prefix}: "
            f"{prefix}_HOST / {prefix}_KEY / {prefix}_MODEL"
        )

    timeout = int(os.environ.get(f"{prefix}_TIMEOUT", "60"))

    return HttpsApi(
        host=host,
        key=key,
        model=model,
        timeout=timeout,
    )





# ===================== 初始化：随机 prompt 种群 =====================

def generate_initial_population(
    teacher_llm: HttpsApi,
    pop_size: int,
    include_baseline: bool = True,
) -> List[RouterPromptCandidate]:
    """
    用一个较强的 teacher LLM 自动生成一批“模型能力描述”，
    尽量覆盖不同的潜在维度（例如：全局规划、局部修补、长上下文跟踪、
    数值稳定性、边界情况处理、风格一致性等等），而不局限在“创意/细心/代码能力强”。
    """
    candidates: List[RouterPromptCandidate] = []
    cid = 0

    if include_baseline:
        # 把你当前使用的静态 router 描述作为一个对照基线
        candidates.append(
            RouterPromptCandidate(
                cid=cid,
                deepseek_desc=DEFAULT_DEEPSEEK_DESC,
                qwen_desc=DEFAULT_QWEN_DESC,
            )
        )
        cid += 1

    system_prompt = (
        "You are designing routing prompts for an automatic heuristic design (AHD) "
        "system based on Evolution of Heuristics (EoH) for online bin packing. "
        "Two backend LLMs are available: DEEPSEEK and QWEN. "
        "You will propose diverse hypotheses about their relative strengths and weaknesses."
    )

    for _ in range(pop_size - len(candidates)):
        user_prompt = f"""
We need a NEW pair of capability descriptions for two LLMs, [DEEPSEEK] and [QWEN],
used inside a routing agent. The agent will see the full AHD prompt (including task
description, operator info like E1/E2/M1/M2, current heuristic code, etc.) and must
choose which backend model is more suitable for THIS call.

Requirements for this particular candidate:
- Explore some *less obvious* capability dimensions (e.g., robustness to noisy code,
  ability to reason about capacity constraints, sensitivity to long-term state, etc.).
- Make DEEPSEEK and QWEN *complementary* rather than symmetric.
- Do NOT talk about API quota, price, or latency.
- Each description should be 4–7 bullet points, short but concrete.

Return ONLY a JSON object with the following fields:

{{
  "deepseek_desc": "markdown bullet list describing DEEPSEEK's strengths/weaknesses",
  "qwen_desc": "markdown bullet list describing QWEN's strengths/weaknesses"
}}
"""
        try:
            raw = teacher_llm.draw_sample(prompt=f"{system_prompt}\n\n{user_prompt}")
        except TypeError:
            raw = teacher_llm.draw_sample(f"{system_prompt}\n\n{user_prompt}")

        parsed = _extract_json_block(raw) or {}
        deepseek_desc = parsed.get("deepseek_desc", DEFAULT_DEEPSEEK_DESC)
        qwen_desc = parsed.get("qwen_desc", DEFAULT_QWEN_DESC)

        candidates.append(
            RouterPromptCandidate(
                cid=cid,
                deepseek_desc=deepseek_desc,
                qwen_desc=qwen_desc,
            )
        )
        cid += 1

    return candidates


def _extract_json_block(text: str) -> Optional[Dict[str, Any]]:
    """
    从 LLM 输出中粗暴地抓取第一个 JSON 块并解析。
    如果失败就返回 None，由调用方做 fallback。
    """
    if not text:
        return None

    s = str(text)
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None

    json_str = s[start : end + 1]
    try:
        return json.loads(json_str)
    except Exception:
        return None


# ===================== 总体优化循环 =====================

def optimize_router_prompts(
    num_generations: int = 3,
    population_size: int = 4,
    num_runs_per_candidate: int = 2,
    log_root: str = "logs_router_learning",
) -> RouterPromptCandidate:
    """
    整体学习循环：
    - 初始化一批 candidate
    - 每一代：
        * 前向：在 EoH+OBP 上评估所有 candidate
        * 构造 best-anchored pairs (h_best, h_i)
        * 反向：用 textual gradient 更新所有非最优个体
    - 返回最后一代中分数最高的 candidate
    """

    # 1. 构造几个 LLM client
    deepseek_llm = build_https_llm_from_env("DEEPSEEK")
    qwen_llm = build_https_llm_from_env("QWEN")

    # 路由 agent 可以和其中一个共用，也可以单独指定 ROUTER_*
    try:
        router_agent_llm = build_https_llm_from_env("ROUTER")
    except ValueError:
        router_agent_llm = qwen_llm

    # teacher LLM 用于生成 / 更新 prompt，一般选择最强的模型
    try:
        teacher_llm = build_https_llm_from_env("TEACHER")
    except ValueError:
        teacher_llm = router_agent_llm

    os.makedirs(log_root, exist_ok=True)

    # 2. 初始化种群
    population = generate_initial_population(
        teacher_llm=teacher_llm,
        pop_size=population_size,
        include_baseline=True,
    )

    return population


# ===================== 一个简单的测试入口 =====================

def test_learned_router(
    prompt_json_path: str,
    num_runs: int = 5,
    log_root: str = "logs_router_learned_eval",
):
    """
    给定已经学到的 best_router_prompt.json，重复多次 EoH，验证性能。

    这个函数基本就是你原来的 multirun_eoh_agent_router.py，
    只是从文件里读取 deepseek_desc / qwen_desc。
    """
    with open(prompt_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    deepseek_desc = data["deepseek_desc"]
    qwen_desc = data["qwen_desc"]

    deepseek_llm = build_https_llm_from_env("DEEPSEEK")
    qwen_llm = build_https_llm_from_env("QWEN")
    try:
        router_agent_llm = build_https_llm_from_env("ROUTER")
    except ValueError:
        router_agent_llm = qwen_llm

    all_scores: List[float] = []
    all_results: List[Dict[str, Any]] = []

    for r in range(1, num_runs + 1):
        cand = RouterPromptCandidate(
            cid=0,
            deepseek_desc=deepseek_desc,
            qwen_desc=qwen_desc,
        )
        result_cand = evaluate_candidate(
            candidate=cand,
            deepseek_llm=deepseek_llm,
            qwen_llm=qwen_llm,
            router_agent_llm=router_agent_llm,
            log_root=os.path.join(log_root, f"run_{r}"),
            num_runs=1,
        )
        all_results.extend(result_cand.meta["runs"])
        if result_cand.score is not None:
            all_scores.append(result_cand.score)

    if all_scores:
        avg_best_score = statistics.mean(all_scores)
    else:
        avg_best_score = None

    total_deepseek = sum(r["deepseek_calls"] for r in all_results)
    total_qwen = sum(r["qwen_calls"] for r in all_results)
    total_calls = total_deepseek + total_qwen

    if total_calls > 0:
        avg_deepseek_ratio = total_deepseek / total_calls
        avg_qwen_ratio = total_qwen / total_calls
    else:
        avg_deepseek_ratio = 0.0
        avg_qwen_ratio = 0.0

    print("\n========== Summary over learned-router runs ==========")
    print(f"Number of runs: {num_runs}")
    print(f"Average best_score: {avg_best_score}")
    print(
        "Average call ratio (aggregated over all runs): "
        f"deepseek={avg_deepseek_ratio:.3f}, qwen={avg_qwen_ratio:.3f}"
    )



best = optimize_router_prompts(
        num_generations=2,
        population_size=4,
        num_runs_per_candidate=1,
        log_root="logs_router_learning_demo",
    )
    
for individual in best:
    print(individual)
    # print("\n>>> Now evaluating the learned router on a few fresh runs...")
    # test_learned_router(
    #     prompt_json_path="logs_router_learning_demo/best_router_prompt.json",
    #     num_runs=3,
    #     log_root="logs_router_learned_eval_demo",
    # )


RouterPromptCandidate(cid=0, deepseek_desc='- Good at understanding MULTIPLE given algorithms and synthesizing high-level patterns across them.\n- Strong global reasoning: can compare different heuristic ideas and design NEW variants that mix them.\n- Comfortable writing relatively long Python code blocks from scratch, including control flow and vectorized NumPy code.\n- Good at exploratory / high-variance proposals that may drastically change the search behaviour of the heuristic.\n', qwen_desc='- Very careful, conservative, and detail-oriented when editing code.\n- Strong at LOCAL edits and REFINEMENT of an existing algorithm:\n  * adjusting weights, coefficients, and thresholds,\n  * changing parameter settings of a score function,\n  * improving robustness, boundary handling, and code safety.\n- Good at reading ONE given algorithm and carefully modifying parts of the logic\n  while keeping the overall structure and invariants intact.\n- Tends to respect types and constraints and av

In [13]:
print(best[0].deepseek_desc)


- Good at understanding MULTIPLE given algorithms and synthesizing high-level patterns across them.
- Strong global reasoning: can compare different heuristic ideas and design NEW variants that mix them.
- Comfortable writing relatively long Python code blocks from scratch, including control flow and vectorized NumPy code.
- Good at exploratory / high-variance proposals that may drastically change the search behaviour of the heuristic.



In [15]:
print(best[1].deepseek_desc[0])

Excels at detecting and correcting subtle logical inconsistencies in heuristic state transitions, especially when constraints evolve over time.
